# 03 Engineered Features and Embeddings

Extract eight engineered feature-group scores and cache frozen DistilBERT CLS-position embeddings for the manuscript fusion ablations.

In [ ]:
from pathlib import Path
import os

# Works from the repository root, from notebooks/, or in Colab after setting BASE_DIR.
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/content/thesis-modeling')]
BASE_DIR = next((p for p in CANDIDATES if (p / 'data' / '06_model_ready').exists()), Path('..')).resolve()
DATA_DIR = BASE_DIR / 'data' / '06_model_ready'
RESULTS_DIR = BASE_DIR / 'results'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
REPORTS_DIR = BASE_DIR / 'reports'
MODELS_DIR = BASE_DIR / 'trained_models'
for d in [RESULTS_DIR, ARTIFACTS_DIR, REPORTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('BASE_DIR =', BASE_DIR)
print('DATA_DIR exists =', DATA_DIR.exists())

import json
import random
import re
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FEATURE_DIR = ARTIFACTS_DIR / 'features'
EMBED_DIR = ARTIFACTS_DIR / 'embeddings'
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
EMBED_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {
    'train_clean': DATA_DIR / 'clean' / 'train_clean.csv',
    'val_clean': DATA_DIR / 'clean' / 'val_clean.csv',
    'test_clean': DATA_DIR / 'clean' / 'test_clean.csv',
    'val_adv_10': DATA_DIR / 'adversarial_validation' / 'val_adv_10.csv',
    'val_adv_20': DATA_DIR / 'adversarial_validation' / 'val_adv_20.csv',
    'test_adv_10': DATA_DIR / 'adversarial_test' / 'test_adv_10.csv',
    'test_adv_20': DATA_DIR / 'adversarial_test' / 'test_adv_20.csv',
    'test_adv_30': DATA_DIR / 'adversarial_test' / 'test_adv_30.csv',
}

URL_RE = re.compile(r'(https?://|www\.|\b[a-z0-9.-]+\.(?:com|net|org|info|co|ly|ph|sg|in)\b)', re.I)
OTP_RE = re.compile(r'\b(?:otp|code|pin|one[- ]?time|verification)\b|\b\d{4,8}\b', re.I)
URGENCY_RE = re.compile(r'\b(?:urgent|immediately|today|now|blocked|suspended|expire|limited|within|verify|update|required)\b', re.I)
FINANCE_RE = re.compile(r'\b(?:bank|account|card|debit|credit|upi|loan|payment|refund|cash|prize|reward|balance|transaction|pan|sbi|yono)\b', re.I)
IDENTITY_RE = re.compile(r'\b(?:kyc|password|login|username|id|identity|ssn|aadhaar|pan|verify|confirm|secure)\b', re.I)
CONTACT_RE = re.compile(r'\b(?:call|sms|reply|contact|whatsapp|helpline|customer care|click|link)\b', re.I)
OBFUSCATION_RE = re.compile(r'[^\x00-\x7F]|[@#$%^*_~=<>]{2,}|\b[a-z]*\d+[a-z\d]*\b|\s{2,}', re.I)
BRAND_RE = re.compile(r'\b(?:sbi|yono|hdfc|icici|axis|paypal|amazon|apple|google|dhl|fedex|singpost|netflix|paytm|gcash)\b', re.I)
FEATURE_NAMES = [
    'g1_url_link', 'g2_otp_verification', 'g3_urgency_pressure', 'g4_financial_terms',
    'g5_identity_security', 'g6_contact_action', 'g7_obfuscation_surface', 'g8_brand_institution'
]
PATTERNS = [URL_RE, OTP_RE, URGENCY_RE, FINANCE_RE, IDENTITY_RE, CONTACT_RE, OBFUSCATION_RE, BRAND_RE]

def extract_features(texts):
    rows = []
    for text in texts:
        s = '' if pd.isna(text) else str(text)
        length_norm = max(len(s), 1) / 160.0
        vals = []
        for pattern in PATTERNS:
            count = len(pattern.findall(s))
            vals.append(min(count / max(length_norm, 1.0), 5.0) / 5.0)
        rows.append(vals)
    return np.asarray(rows, dtype=np.float32)

for split_name, path in SPLITS.items():
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df['model_text'] = df['model_text'].fillna('').astype(str)
    features = extract_features(df['model_text'])
    np.save(FEATURE_DIR / f'{split_name}_features.npy', features)
    meta = df[['final_row_id', 'label_id', 'normalized_label']].copy()
    meta.to_csv(FEATURE_DIR / f'{split_name}_feature_rows.csv', index=False)

(FEATURE_DIR / 'feature_metadata.json').write_text(json.dumps({
    'feature_names': FEATURE_NAMES,
    'description': 'Eight deterministic engineered feature-group scores used for frozen DistilBERT fusion ablations.',
    'source_text_column': 'model_text',
}, indent=2), encoding='utf-8')
print('Saved engineered features for', len(SPLITS), 'splits')

import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = 'distilbert-base-cased'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

@torch.no_grad()
def embed_texts(texts, batch_size=64, max_length=128):
    all_embeddings = []
    for start in tqdm(range(0, len(texts), batch_size), desc='Embedding'):
        batch = list(texts[start:start + batch_size])
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)
        cls = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy().astype(np.float32)
        all_embeddings.append(cls)
    return np.vstack(all_embeddings)

for split_name, path in SPLITS.items():
    out_path = EMBED_DIR / f'{split_name}_distilbert_cls.npy'
    if out_path.exists():
        arr = np.load(out_path)
        print(split_name, 'embedding exists', arr.shape)
        continue
    df = pd.read_csv(path)
    texts = df['model_text'].fillna('').astype(str).tolist()
    embeddings = embed_texts(texts)
    np.save(out_path, embeddings)
    print(split_name, embeddings.shape)

(EMBED_DIR / 'embedding_metadata.json').write_text(json.dumps({
    'model_name': MODEL_NAME,
    'embedding': 'last_hidden_state[:, 0, :] frozen DistilBERT CLS-position representation',
    'max_length': 128,
    'device': str(device),
}, indent=2), encoding='utf-8')
